In [1]:
# !pip install mljar-supervised
# from supervised import AutoML
# !pip install autogluon
# from autogluon.tabular import TabularPredictor
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
import xgboost as xgb
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns 
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import BaggingClassifier,RandomForestClassifier,ExtraTreesClassifier,GradientBoostingClassifier,HistGradientBoostingClassifier,AdaBoostClassifier
from sklearn.ensemble import VotingClassifier,StackingClassifier
from sklearn.neighbors import KNeighborsClassifier
from catboost import CatBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# from sklearn.metrics import (    accuracy_score,    confusion_matrix,f1_score,precision_score, recall_score,roc_auc_score)
# from sklearn.metrics import make_scorer
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold,cross_val_score, StratifiedShuffleSplit
from sklearn.cluster import KMeans
from sklearn.preprocessing import OneHotEncoder, LabelEncoder,FunctionTransformer, StandardScaler
from category_encoders import TargetEncoder, CatBoostEncoder
from category_encoders.binary import BinaryEncoder
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold, SelectFromModel,SelectPercentile
from statsmodels.stats.outliers_influence import variance_inflation_factor
pd.set_option('display.max_rows', 500)
import os
import shap
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/life-resorce/sample_submission.csv
/kaggle/input/life-resorce/train.csv
/kaggle/input/life-resorce/test.csv


In [20]:
def preprocess(data):
    # 모든 값이 동일한 열 찾기
    constant_columns = []
    for col in data.columns:
        if data[col].nunique() == 1 and data[col].isnull().sum() == 0:
            constant_columns.append(col)

    # 모든 값이 결측인 열 찾기
    no_values = []
    for col in data.columns:
        if data[col].isnull().all():
            no_values.append(col)

    columns_to_remove = constant_columns + no_values 

    # data에서 해당 열 제거
    data = data.drop(columns=columns_to_remove)
    return data
class FeatureSelect:
    """
        변수 선택 기능 
        kbest, vif, variance, model_importance 
    """
    def __init__(self, x: pd.DataFrame, y: pd.Series, test: pd.DataFrame, ratio: float, 
                 vif_threshold: float, variance_threshold: float, model=None):
        self.x = x
        self.y = y
        self.test = test
        self.ratio = ratio
        self.vif_threshold = vif_threshold
        self.var_threshold = variance_threshold
        self.model = model
        
        # 선택할 피처의 수
        self.select_nums = max(1, int(len(x.columns) * ratio))  # 최소 1개 피처 선택
        self.kbest_selector = SelectKBest(f_classif, k=self.select_nums)
        self.variance_selector = VarianceThreshold(threshold=self.var_threshold)

    def _check_data(self):
        """데이터 유효성 검사"""
        if self.x.isnull().values.any() or self.test.isnull().values.any():
            raise ValueError("Input data contains null values.")
        if self.x.shape[0] != self.y.shape[0]:
            raise ValueError("The number of samples in x and y must be equal.")

    def kbest_select(self) -> tuple:
        """SelectKBest 피처 선택"""
        self._check_data()
        self.kbest_selector.fit(self.x, self.y)
        kbest_columns = self.kbest_selector.get_support(indices=True)
        self.x = self.x.iloc[:, kbest_columns]
        self.test = self.test.iloc[:, kbest_columns]
        print("KBest selection completed.")
        return self.x, self.test

    def vif_select(self) -> tuple:
        """VIF 기반 피처 선택"""
        self._check_data()

        # VIF 계산
        vif = pd.DataFrame()
        vif['Features'] = self.x.columns
        vif['VIF'] = [
            variance_inflation_factor(self.x.values, i) for i in tqdm(range(self.x.shape[1]), desc="Calculating VIF")
        ]
        vif['VIF'] = round(vif['VIF'], 2)
        vif = vif.sort_values(by="VIF", ascending=False)
        print("VIF values (sorted):")
        print(vif)

        # VIF 기준으로 제거할 피처 찾기
        features_to_remove = vif[vif['VIF'] > self.vif_threshold]['Features'].tolist()
        x_selected = self.x.drop(columns=features_to_remove)
        test_selected = self.test.drop(columns=features_to_remove)

        if features_to_remove:            print(f"Removed features with VIF > {self.vif_threshold}: {features_to_remove}")
        else:            print("No features removed based on VIF.")
            
        # VIF 시각화
        plt.figure(figsize=(10, 6))
        plt.barh(vif['Features'], vif['VIF'], color='skyblue')
        plt.axvline(x=self.vif_threshold, color='red', linestyle='--', label=f'VIF Threshold = {self.vif_threshold}')
        plt.title("VIF Values for Features")
        plt.xlabel("VIF")
        plt.ylabel("Features")
        plt.legend()
        plt.show()
        return x_selected, test_selected
    
    def variancethreshold_select(self) -> tuple:
        """VarianceThreshold 피처 선택"""
        self.variance_selector.fit(self.x)
        selected_cols = self.variance_selector.get_support(indices=True)
        self.x = self.x.iloc[:, selected_cols]
        self.test = self.test.iloc[:, selected_cols]
        print("VarianceThreshold selection completed.")
        return self.x, self.test

    def model_select(self, model_threshold: float = None) -> tuple:
        """모델 기반 피처 선택"""
        if self.model is None:          raise ValueError("No model provided for feature selection.")
            
        # 모델에 기반한 피처 선택
        model_selector = SelectFromModel(self.model, threshold = model_threshold)
        model_selector.fit(self.x, self.y)
        # 선택된 피처의 마스크
        selected_features_mask = model_selector.get_support()
        # 선택된 피처와 삭제된 피처 계산
        selected_features = self.x.columns[selected_features_mask]
        removed_features = self.x.columns[~selected_features_mask]
        x_selected = self.x.loc[:, selected_features_mask]
        test_selected = self.test.loc[:, selected_features_mask]

        # 삭제된 피처 정보 출력
        if removed_features.any():
            print(f"Removed features: {removed_features.tolist()}")
            print(f"Number of features removed: {len(removed_features)}")
        else: print("No features removed.")

        print("Model-based feature selection completed.")

        return x_selected, test_selected

    def model_feature_importance(self):
        """모델 피처 중요도 출력 및 시각화"""
        if self.model is None:
            print("No model is fitted yet.")
            return

        # 모델 학습
        self.model.fit(self.x, self.y)
        importances = self.model.feature_importances_

        # 피처 수 확인
        if len(importances) == 0:
            print("No feature importances found.")
            return

        # 피처 중요도 정렬
        indices = np.argsort(importances)[::-1]
        print("Feature ranking:")
        for f in range(len(indices)):print(f"{f + 1}. feature {self.x.columns[indices[f]]} ({importances[indices[f]]:.6f})")

        # 중요도 시각화
        plt.figure(figsize=(20, 10))
        plt.title("Feature Importances")
        plt.bar(range(len(importances)), importances[indices], align='center')
        plt.xticks(range(len(importances)), self.x.columns[indices], rotation=90)
        plt.xlim([-1, len(importances)])
        plt.ylabel("Importance")
        plt.xlabel("Features")
        plt.tight_layout()
        plt.show()

    def percentile_select(self, percentile: float) -> tuple:
        """SelectPercentile 피처 선택"""
        self._check_data()
        
        # SelectPercentile 초기화 및 피팅
        self.percentile_selector = SelectPercentile(f_classif, percentile=percentile)
        self.percentile_selector.fit(self.x, self.y)
        
        selected_cols = self.percentile_selector.get_support(indices=True)
        self.x = self.x.iloc[:, selected_cols]
        self.test = self.test.iloc[:, selected_cols]
        
        print(f"Percentile selection completed (top {percentile}%).")
        return self.x, self.test
    
class DataEncoder:
    """
    Binary, Onehot, Label, Target, KfoldTarget, Catboost 
    """
    def __init__(self, train, test, target, binary_cols=None, onehot_cols=None, catboost_cols=None, label_cols=None, base_target_cols=None, target_cols=None):
        self.train = train.copy()
        self.test = test.copy()
        self.target = target
        self.binary_cols = binary_cols
        self.onehot_cols = onehot_cols
        self.label_cols = label_cols
        self.target_cols = target_cols
        self.catboost_cols = catboost_cols
        self.base_target_cols = base_target_cols

    def onehot_encode(self):
        if self.onehot_cols is None:
            return self
        onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        tmp = pd.DataFrame(
            onehot_encoder.fit_transform(self.train[self.onehot_cols]),
            columns=onehot_encoder.get_feature_names_out()
        )
        self.train = pd.concat([self.train, tmp], axis=1).drop(columns=self.onehot_cols)
        tmp = pd.DataFrame(
            onehot_encoder.transform(self.test[self.onehot_cols]),
            columns=onehot_encoder.get_feature_names_out()
        )
        self.test = pd.concat([self.test, tmp], axis=1).drop(columns=self.onehot_cols)
        return self

    def label_encode(self):
        if self.label_cols is None:            return self
        for col in self.label_cols:
            le = LabelEncoder()
            le.fit(self.train[col])
            self.train[col] = le.transform(self.train[col])

            # 새로운 라벨 처리를 위한 딕셔너리 생성
            le_dict = dict(zip(le.classes_, le.transform(le.classes_)))

            # 테스트 데이터에 대해 라벨 변환
            self.test[col] = self.test[col].apply(lambda x: le_dict.get(x, -1))  # 새로운 값에 대해 -1로 변환

        return self

    def target_encode(self, smoothing=1):
        if self.base_target_cols is None:
            return self
        encoder = TargetEncoder(smoothing=smoothing)
        for col in self.base_target_cols:
            self.train[col] = encoder.fit_transform(self.train[col], self.train[self.target])
            self.test[col] = encoder.transform(self.test[col])
        return self

    def kfold_target_encode(self, n_splits=5, smoothing=1, min_samples_leaf=1, remove_original_col=True):
        if self.target_cols is None:
            return self
        for col in self.target_cols:
            target = self.train[self.target].values
            kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
            oof = np.zeros(len(self.train))
            for tr_ind, val_ind in kf.split(self.train):
                x_tr, x_val = self.train.iloc[tr_ind], self.train.iloc[val_ind]
                means = x_val[col].map(x_tr.groupby(col)[self.target].mean())
                oof[val_ind] = means
            smoothing_factor = 1 / (1 + np.exp(-(oof - min_samples_leaf) / smoothing))
            oof_smooth = smoothing_factor * oof + (1 - smoothing_factor) * target.mean()
            self.train[col + '_TE'] = oof_smooth
            self.test[col + '_TE'] = self.test[col].map(self.train.groupby(col)[self.target].mean())
            if remove_original_col:
                self.train.drop(columns=[col], inplace=True)
                self.test.drop(columns=[col], inplace=True)
        return self

    def catboost_encode(self):
        if self.catboost_cols is None:
            return self
        enc = CatBoostEncoder(cols=self.catboost_cols)
        enc.fit(self.train[self.catboost_cols], self.train[self.target])
        self.train[self.catboost_cols] = enc.transform(self.train[self.catboost_cols])
        self.test[self.catboost_cols] = enc.transform(self.test[self.catboost_cols])
        return self

    def binary_encode(self):
        if self.binary_cols is None:
            return self
        additional_cols = [col for col in self.train.columns if col not in self.test.columns]
        train_temp = self.train.drop(columns=additional_cols)
        enc = BinaryEncoder(cols=self.binary_cols)
        train_enc = enc.fit_transform(train_temp)
        test_enc = enc.transform(self.test)
        for col in additional_cols:
            train_enc[col] = self.train[col]
        self.train = train_enc
        self.test = test_enc
        return self
    


class Shap:
    def __init__(self, model, X_train):
        """
        :param model: 학습된 모델
        :param X_train: 학습에 사용된 데이터 (SHAP 계산에 필요)
        """
        self.model = model
        self.X_train = X_train
        self.explainer = shap.TreeExplainer(self.model)
        self.shap_values = self.explainer.shap_values(self.X_train)
    
    def plot_shap(self):
        """SHAP을 이용한 특성 중요도 시각화를 수행합니다."""
        if self.shap_values is None:
            raise ValueError("SHAP 값을 먼저 계산하세요. calculate_shap_values()를 호출하세요.")
        
        # SHAP Summary Plot
        shap.summary_plot(self.shap_values, self.X_train)
    
    def get_top_features(self, threshold=0.05):
        """
        중요도가 특정 임계치보다 높은 특성들을 반환합니다.
        
        :param threshold: 중요도 임계치 (기본값은 0.05)
        :return: 임계치보다 높은 중요도를 가진 특성들
        """
        if self.shap_values is None:
            raise ValueError("SHAP 값을 먼저 계산하세요. calculate_shap_values()를 호출하세요.")
        
        # 특성 중요도 계산 (평균 절대 SHAP 값 기준)
        feature_importance = np.abs(self.shap_values).mean(axis=0)
        
        # 임계치보다 높은 중요도를 가진 특성들의 인덱스와 이름 반환
        important_features = [self.X_train.columns[i] for i in np.where(feature_importance > threshold)[0]]
        return important_features
    
    def print_shap(self):
        """특성 중요도를 출력합니다."""
        if self.shap_values is None:
            raise ValueError("SHAP 값을 먼저 계산하세요. calculate_shap_values()를 호출하세요.")
        
        feature_importance = np.abs(self.shap_values).mean(axis=0)
        feature_names = self.X_train.columns
        
        # 중요도 순으로 정렬하여 출력
        importance_order = np.argsort(feature_importance)[::-1]
        for idx in importance_order:
            print(f"{feature_names[idx]}: {feature_importance[idx]}")


In [24]:
train = pd.read_csv("/kaggle/input/life-resorce/train.csv").drop(columns = ['ID'])
test = pd.read_csv("/kaggle/input/life-resorce/test.csv").drop(columns = ['ID'])
train= preprocess(train)
cols =list(train.columns); cols.remove('SUBCLASS') ; test = test[cols]

In [25]:
# SUBCLASS 가 범주형이기 때문에 LabelEncoder 사용
le_subclass = LabelEncoder()
train['SUBCLASS'] = le_subclass.fit_transform(train['SUBCLASS'])
print("SUBCLASS 인코딩 완료 ")
# encoding 
non_num_cols = train.select_dtypes(exclude='number').columns
enc = DataEncoder(train=train,test =test ,target='SUBCLASS',catboost_cols=non_num_cols)
data =enc.catboost_encode()
train,test =data.train,data.test 
x_train = train.drop(columns=['SUBCLASS'])
y_subclass = train['SUBCLASS'] 
x_test = test.copy()

SUBCLASS 인코딩 완료 


In [36]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# 데이터 정규화
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

# Tensor로 변환
x_train_tensor = torch.tensor(x_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_subclass, dtype=torch.long)
x_test_tensor = torch.tensor(x_test, dtype=torch.float32)
# y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# 배치 크기 정의
batch_size = 128
# DataLoader 정의
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# MLP 모델 정의
class MLP(nn.Module):
    def __init__(self, input_size, num_classes):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, num_classes)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.fc4(x)
        return x

# 모델, 손실 함수, 옵티마이저 정의
input_size = x_train.shape[1]
num_classes = 26
model = MLP(input_size, num_classes)
# CUDA 사용 가능 시 모델을 GPU로 전송
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [37]:

# 학습 함수 정의
def train_model(model, train_loader, criterion, optimizer, num_epochs=100):
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)  # 데이터를 GPU로 전송
            
            # 모델에 입력하고 역전파 및 최적화
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
        
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")
# 모델 학습
train_model(model, train_loader, criterion, optimizer)

Epoch 1/100, Loss: 2.8980
Epoch 2/100, Loss: 2.0596
Epoch 3/100, Loss: 1.4021
Epoch 4/100, Loss: 1.0242
Epoch 5/100, Loss: 0.8152
Epoch 6/100, Loss: 0.7253
Epoch 7/100, Loss: 0.6581
Epoch 8/100, Loss: 0.6132
Epoch 9/100, Loss: 0.5748
Epoch 10/100, Loss: 0.5620
Epoch 11/100, Loss: 0.5363
Epoch 12/100, Loss: 0.5676
Epoch 13/100, Loss: 0.5184
Epoch 14/100, Loss: 0.4886
Epoch 15/100, Loss: 0.4761
Epoch 16/100, Loss: 0.5309
Epoch 17/100, Loss: 0.4932
Epoch 18/100, Loss: 0.4522
Epoch 19/100, Loss: 0.4344
Epoch 20/100, Loss: 0.4144
Epoch 21/100, Loss: 0.3991
Epoch 22/100, Loss: 0.4042
Epoch 23/100, Loss: 0.4277
Epoch 24/100, Loss: 0.4173
Epoch 25/100, Loss: 0.3835
Epoch 26/100, Loss: 0.3699
Epoch 27/100, Loss: 0.3689
Epoch 28/100, Loss: 0.3602
Epoch 29/100, Loss: 0.3442
Epoch 30/100, Loss: 0.3348
Epoch 31/100, Loss: 0.3291
Epoch 32/100, Loss: 0.3199
Epoch 33/100, Loss: 0.3324
Epoch 34/100, Loss: 0.3253
Epoch 35/100, Loss: 0.3379
Epoch 36/100, Loss: 0.3123
Epoch 37/100, Loss: 0.3665
Epoch 38/1

In [38]:

# 추론 함수 정의
def infer_model(model, x_test_tensor):
    model.eval()
    with torch.no_grad():
        x_test_tensor = x_test_tensor.to(device)  # 데이터를 GPU로 전송
        outputs = model(x_test_tensor)
        _, predictions = torch.max(outputs, 1)
    return predictions.cpu().numpy()

# 테스트 데이터로 추론
predictions = infer_model(model, x_test_tensor)

In [39]:
original_labels = le_subclass.inverse_transform(predictions)

In [40]:
submisson = pd.read_csv("/kaggle/input/life-resorce/sample_submission.csv")
submisson["SUBCLASS"] = original_labels
submisson.to_csv('./mlp2.csv', encoding='UTF-8-sig', index=False)